In [1]:
import pandas as pd
import requests
from io import StringIO
import re
#!pip install ipykernel
#!pip install lxml
#!pip install openpyxl
#!pip install pyarrow

# SHFE prices

In [16]:
response = requests.get (f"https://www.shfe.com.cn/data/tradedata/future/dailydata/kx20260522.dat")

In [17]:
data = response.json()
df = pd.DataFrame(data['o_curinstrument'])

df["product_group_index"] = pd.factorize(df["PRODUCTGROUPID"])[0] # Create a new column with the group index

df = df.sort_values(["product_group_index", "DELIVERYMONTH"], ascending=[True, True], kind='stable') # Sort by group index and delivery month
df.drop(columns=["product_group_index"], inplace=True) # Remove the temporary group index column
df.reset_index(drop=True, inplace=True) # Reset the index after sorting

df.insert(0,"M",df.groupby("PRODUCTGROUPID").cumcount() + 1) # Add a new column "M" with the cumulative count of rows within each group, starting from 1
sub_totals_index = df.groupby("PRODUCTGROUPID").tail(1).index # Get the index of the last row of each group which turns to be the subtotal row
df.loc[sub_totals_index, "M"] = 0 # Set the "M" value to 0 for the subtotal rows

In [18]:
df.to_csv("shfe_prices.csv", index=False, encoding="utf-8-sig")

# SHFE stocks

In [39]:
response = requests.get(f"https://www.shfe.cn/data/tradedata/future/stockdata/weeklystock_20260522/EN/all.html")
data = pd.read_html(StringIO(response.text)) # A list of DataFrames is crated, one for each table found in the HTML

In [43]:
DF = pd.DataFrame() # Create an empty DataFrame to store the concatenated results
for df in data:
    if isinstance(df.columns, pd.MultiIndex): # Flatten the df with MultiIndex columns
        df.columns = [col[0] if col[0] == col [1] else f"{col[0]} ({col[1]})" for col in df.columns]
    
    if "Change" in df.columns: # If 'Change' column exists, rename it to include the previous column's sufix in parentheses
        ls = df.columns.to_list()
        i = ls.index("Change")
        df.columns.values[i] = f"Change {ls[i-1][ls[i-1].find('('):ls[i-1].find(')')+1]}"
    renames= {
        "Theoretical Available Capacity (Last week)": "Storage Capacity (Last week)",
        "Theoretical Available Capacity (This Week)": "Storage Capacity (This Week)",
        "Theoretical Available Capacity (Change)": "Storage Capacity (Change)",
        "Storage of last week": "Previous Week (Delivery-able)",
        "Storage of this week": "This Week (Delivery-able)",
        "Storage Change": "Change (Delivery-able)",
        "Storage of last week (Delivery-able)": "Previous Week (Delivery-able)",
        "Storage of last week (On Warrant)": "Previous Week (On Warrant)",
        "Storage of this week (Delivery-able)": "This Week (Delivery-able)",
        "Storage of this week (On Warrant)": "This Week (On Warrant)",
        "Storage Change (Delivery-able)": "Change (Delivery-able)",
        "Storage Change (On Warrant)": "Change (On Warrant)",
        "Factory Warehouse" : "Warehouse",
        "Depot" : "Warehouse",
        "Grade" : "Crude",
        "Factory Depot" : "Warehouse"}
    df.rename(columns=renames, inplace=True) # Rename columns as per the 'renames' dictionary, if they exist.
    
    u = df.iloc[0,-1] # Get the last column of the first row which may contain the "Unit: " information
    if isinstance(u, str) and "Unit：" in u: # If the "Unit: " measurement is provided in the last column of the first row...
        df.insert(0, "Commodity", df.iloc[0,0]) #...Create a 'Commodity' column by extracting it from the first row's first column
        df.insert(1, "Unit", u.split("Unit：")[1]) #...Create a 'Unit' column by extracting it
        df.drop(df.index[0], inplace=True) # Drop the first row which is now redundant after extracting the 'Unit' and 'Commodity' info.
        df.replace("--", pd.NA, inplace=True) #...Replace any occurrence of "--" with NaN
    
    # Convert columns that contain numeric data stored as strings to numeric types, coercing errors to NaN
    df = df.apply(lambda col: pd.to_numeric(col, errors='coerce') if col.dropna().astype(str).str.fullmatch(r"-?\d+(\.\d+)?").all() else col)
    
    if df.shape[1] > 1: # If the DataFrame has more than one column...
        DF = pd.concat([DF, df], ignore_index=True) # Concatenate the cleaned DataFrame to the main 'DF'

In [46]:
DF.to_csv("shfe_stocks_final.csv", index=False)
DF.to_pickle("shfe_stocks_final.pkl")
DF.to_hdf("shfe_stocks_final.h5", key="DF", mode="w")
DF.to_parquet("shfe_stocks_final.parquet", index=False, compression="zstd")

In [24]:
text = "Issue No.(19),2026,(Total Issues1386) DATE:2026-05-22"

m = re.search(
    r'Issue No\.\((\d+)\),(\d+),\(Total Issues(\d+)\) DATE:(\d{4})-(\d{2})-(\d{2})',
    text
)

result = f"{m.group(4)}.{m.group(5)}.{m.group(6)} No.{m.group(1)} {m.group(3)} Issues"

print(result)
print (m.group(1), m.group(2), m.group(3), m.group(4), m.group(5), m.group(6))

2026.05.22 No.19 1386 Issues
19 2026 1386 2026 05 22
